# Bringing PCA, t-SNE, and Clustering Together (Exercise)

In the previous notebooks you used **PCA** to compress a dataset (notebook 1) and **t-SNE** to project high-dimensional data into two dimensions (notebook 2). This optional exercise brings those ideas together and adds a first look at **clustering**, which you will study in depth in the clustering repo.

You will work with a real dataset: **Human Activity Recognition (HAR)**. Thirty people wore a smartphone on the waist while doing six everyday activities (walking, walking upstairs, walking downstairs, sitting, standing, and lying down). The phone's accelerometer and gyroscope readings were processed into **561 features** per recording. Your task is to *group* the recordings without using the activity labels, and only afterwards use the labels to check how good the groups are.

The workflow is the standard one: **reduce with PCA, cluster, then visualise the clusters.** With 561 features, the "reduce" step is no longer optional.

## Learning Objectives

By the end of this notebook, you should be able to:

- Apply K-Means to the full 561-feature data and evaluate the clusters against the true activities using the adjusted Rand index and the silhouette score.
- Calculate how many principal components are needed to retain about 90% of the variance.
- Compare clustering quality, silhouette, and runtime at the full dimensionality, the reduced dimensionality, and two dimensions.
- Compare K-Means, agglomerative, and DBSCAN clustering on the reduced data.
- Visualise the activities in two dimensions with t-SNE, and justify why two components are enough for a picture but not for the finest clustering.

## Why reduce dimensions before clustering?

Clustering algorithms group points by **distance**, and distance becomes less informative as the number of features grows: in hundreds of dimensions points drift towards being equally far apart (the *curse of dimensionality*), so the groups blur together. With 561 features, reducing first is what makes clustering practical. PCA helps in three ways here:

- It **removes correlated and low-variance directions** (mostly noise), which can keep the cluster quality while using far fewer features.
- It **speeds up** the clustering, because the algorithm works in far fewer dimensions.
- It lets you **plot the result in 2D**.

A note on scaling: in notebook 1 you standardised the wine features because they were on very different scales. The HAR features are already normalised to the range -1 to 1, so you can go straight to PCA here. Deciding whether to scale is part of the skill: scale when features live on different scales, leave the data as is when they already share one.

The workflow you will follow:

```mermaid
flowchart LR
    A[HAR: 561 features] --> B[PCA: reduce to the informative components]
    B --> C[Cluster: K-Means / hierarchical / DBSCAN]
    C --> D[Visualise: t-SNE in 2D]
```

Two measures tell you how good a clustering is:

- **Adjusted Rand index (ARI)**: compares the cluster assignments with the true labels (1.0 is a perfect match, 0.0 is random). This needs labels, so it is only available here because the activities are labelled.
- **Silhouette score**: measures how compact and well-separated the clusters are, from the data alone (no labels needed). Higher is better.

### Where this is used

The scale, reduce, cluster, visualise workflow is used across industry wherever data is high-dimensional and unlabelled:

- **Wearables and health monitoring**: grouping sensor readings into activities or detecting unusual movement, exactly the HAR setting.
- **Genomics**: single-cell RNA-seq measures thousands of genes per cell, then reduces with PCA and clusters the cells into types (the standard Scanpy and Seurat pipelines).
- **Customer analytics**: reducing many behavioural features before segmenting customers for marketing.
- **Fraud and anomaly detection**: projecting transactions with PCA and flagging points that do not fit the main structure.

The dataset changes, the recipe does not.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, silhouette_score

sns.set()

> [!NOTE]
> The next cell downloads the HAR dataset from OpenML the first time you run it (a few seconds) and caches it locally, so later runs are instant.

The activity labels are the digits 1 to 6: 1 walking, 2 walking upstairs, 3 walking downstairs, 4 sitting, 5 standing, 6 lying down. We also build a smaller random sample now, which you will use for the slower steps (t-SNE and DBSCAN) so they stay fast.

In [ ]:
har = fetch_openml("har", version=1, as_frame=True)
X = har.data.to_numpy()
y = har.target.to_numpy().astype(int)

# A 2,500-row random sample for the slower steps (Steps 4 and 5)
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X), size=2500, replace=False)
X_sample, y_sample = X[sample_idx], y[sample_idx]

X.shape, np.unique(y)

## Step 1: Cluster the full 561-feature data

Cluster the recordings into 6 groups with K-Means on the full data, then measure the result against the true activities. This is your baseline.

Use `KMeans(n_clusters=6, random_state=42, n_init=10)`.

In [ ]:
labels_full = KMeans(n_clusters=6, random_state=42, n_init=10).fit_predict(X)
ari_full = adjusted_rand_score(y, labels_full)
sil_full = silhouette_score(X, labels_full)
print(f"full (561 dims): ARI={ari_full:.3f}, silhouette={sil_full:.3f}")

**Explanation**

Clustering the raw 561-dimensional data gives an ARI of about 0.461 against the true activities and a silhouette of about 0.133. This is the baseline the reduced versions are compared with.

## Step 2: How many components carry the variance?

Fit a full PCA on `X` and look at the **cumulative explained variance ratio**. Find the smallest number of components that together retain at least 90% of the variance, and plot the curve so you can see how quickly the variance accumulates. With 561 features, notice how few components you actually need.

In [ ]:
pca_full = PCA().fit(X)
cumulative = np.cumsum(pca_full.explained_variance_ratio_)
n_components_90 = int(np.argmax(cumulative >= 0.90) + 1)
print(f"{n_components_90} components retain 90% of the variance")

plt.plot(range(1, len(cumulative) + 1), cumulative)
plt.axhline(0.90, color="red", linestyle="--", label="90% variance")
plt.axvline(n_components_90, color="grey", linestyle=":")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.legend()
plt.show()

**Explanation**

Only about 36 of the 561 components are needed to keep 90% of the variance, so most of the 561 features are redundant or noise. The curve rises steeply at first and then flattens, which is the signature of data that reduces well.

## Step 3: Cluster at three dimensionalities

Now compare clustering at three sizes: the full **561** features, the reduced **`n_components_90`** features, and just **2** components. For each, run the same K-Means, and record the adjusted Rand index, the silhouette score, and the time the `fit` takes (use `time.time()` around the call).

Before you run it, predict: which will give the best ARI, and which will be fastest? Will 2 components be enough?

In [ ]:
for n_dims in [561, n_components_90, 2]:
    data = X if n_dims == 561 else PCA(n_components=n_dims, random_state=42).fit_transform(X)
    start = time.time()
    labels = KMeans(n_clusters=6, random_state=42, n_init=10).fit_predict(data)
    elapsed = time.time() - start
    ari = adjusted_rand_score(y, labels)
    sil = silhouette_score(data, labels)
    print(f"{n_dims:3d} dims: ARI={ari:.3f}, silhouette={sil:.3f}, time={elapsed:.2f}s")

**Explanation**

At 561 and at 36 dimensions the ARI is identical (about 0.461), but 36 dimensions runs roughly 12 times faster and has a higher silhouette: reducing removed noise without losing the clustering. At 2 dimensions the ARI falls to about 0.299 even though the silhouette is highest, so 2 components are great for plotting but too few to separate all six activities.

## Step 4: Compare three clustering algorithms

Reduce the **sample** (`X_sample`) to 2 components with PCA, then compare three algorithms you will meet in the clustering repo:

- `KMeans(n_clusters=6, random_state=42, n_init=10)`
- `AgglomerativeClustering(n_clusters=6)` (hierarchical)
- `DBSCAN(eps=1.0, min_samples=10)` (density based: it chooses the number of clusters itself and labels sparse points as noise, `-1`)

For each, compute the ARI against `y_sample` and count how many clusters it found.

> [!TIP]
> DBSCAN is sensitive to `eps` (the neighbourhood radius). There is no universal value: the usual way to choose it is a k-distance plot (sort every point's distance to its k-th nearest neighbour and look for the elbow). The `eps=1.0` here is a reasonable starting point for this 2D data, but try a few values and watch how the number of clusters and the noise count change.

In [ ]:
X_sample_pca = PCA(n_components=2, random_state=42).fit_transform(X_sample)

for name, model in [
    ("KMeans", KMeans(n_clusters=6, random_state=42, n_init=10)),
    ("Agglomerative", AgglomerativeClustering(n_clusters=6)),
    ("DBSCAN", DBSCAN(eps=1.0, min_samples=10)),
]:
    labels = model.fit_predict(X_sample_pca)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    ari = adjusted_rand_score(y_sample, labels)
    print(f"{name:14s}: ARI={ari:.3f}, clusters found={n_clusters}")

**Explanation**

On the 2D data K-Means (ARI about 0.313) and agglomerative clustering (about 0.294) both split the sample into six groups, while DBSCAN finds only two clusters (ARI about 0.331) plus some noise. DBSCAN groups by density, so it locks onto the two obvious blobs (moving versus still) rather than the six labelled activities.

## Step 5: Visualise with t-SNE

Project the **sample** (`X_sample`) to 2 dimensions with t-SNE (`init="pca"`, `random_state=42`), then draw the embedding twice side by side: once coloured by the **true activity** `y_sample`, once coloured by a **cluster label** (for example the K-Means labels from Step 4). This shows how well the clusters line up with the real activities.

t-SNE on 2,500 points takes a few seconds, so run this cell once and wait for it.

In [ ]:
embedding = TSNE(n_components=2, init="pca", random_state=42).fit_transform(X_sample)
kmeans_labels = KMeans(n_clusters=6, random_state=42, n_init=10).fit_predict(X_sample_pca)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc = axes[0].scatter(embedding[:, 0], embedding[:, 1], c=y_sample, cmap="tab10", s=8)
axes[0].set_title("t-SNE coloured by true activity")
axes[1].scatter(embedding[:, 0], embedding[:, 1], c=kmeans_labels, cmap="tab10", s=8)
axes[1].set_title("t-SNE coloured by K-Means cluster")
fig.colorbar(sc, ax=axes[0], label="activity")
plt.show()

**Explanation**

The t-SNE map shows a clear split between the moving activities and the still ones, and the K-Means colours line up with that structure. Activities that feel similar to a waist sensor, such as sitting and standing, sit close together or overlap, which is why the six-way ARI is only moderate.

## Check Your Understanding

Answer these in your own words, using the numbers your notebook produced.

**1. How did the ARI at `n_components_90` compare with the full 561 features, and how did the runtime compare? Was reducing the dimensions worth it?**

<details><summary>Show answer</summary>

The 36 components that retain 90% of the variance give the same ARI as the full 561 features (about 0.461), with a slightly higher silhouette, and the K-Means fit runs roughly 12 times faster (about 0.2s versus 2.4s). So yes: reducing kept the clustering quality and made it much cheaper.

</details>

**2. Two components gave a high silhouette but a lower ARI than the reduced set. What does that tell you about the difference between clustering the data and drawing a tidy 2D picture of it?**

<details><summary>Show answer</summary>

At 2 components the silhouette is highest (about 0.444) but the ARI drops to about 0.299. Silhouette only measures how compact and separated the clusters are, not whether they match the true activities. A tidy 2D blob can score well on silhouette while merging activities that ARI counts as wrong, so a good-looking picture is not the same as a good clustering.

</details>

**3. Did DBSCAN find six clusters? The six activities fall into two natural groups (moving vs still). Which grouping did DBSCAN pick up, and why might a density-based method do that?**

<details><summary>Show answer</summary>

No. DBSCAN found two clusters (plus a handful of noise points), matching the moving activities (the walking variants) versus the still ones (sitting, standing, lying). Density-based methods find however many dense regions exist rather than a fixed number, so they lock onto the two clear density groups instead of separating all six activities.

</details>

**4. In the t-SNE plot, some activities (for example sitting and standing) may sit almost on top of each other. What does that suggest, and why should you not read too much into the exact distances t-SNE shows?**

<details><summary>Show answer</summary>

From a waist-mounted sensor, a still upright body produces similar readings whether the person is sitting or standing, so those activities genuinely overlap in feature space and t-SNE places them together. Read the plot only as a rough guide: t-SNE distorts cluster sizes and between-cluster distances by design, so the exact gaps and areas are not meaningful.

</details>

## References & Further Reading

- [**Human Activity Recognition Using Smartphones (UCI)**](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones): The source dataset, with how the 561 features were built from the phone sensors.
- [**scikit-learn: A demo of K-Means clustering on the handwritten digits data**](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_digits.html): Reference example for combining PCA with K-Means and measuring the result.
- [**scikit-learn: adjusted_rand_score**](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html): Comparing a clustering against known labels.
- [**scikit-learn: silhouette_score**](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html): Measuring cluster compactness and separation without labels.
- [**How to Use t-SNE Effectively**](https://distill.pub/2016/misread-tsne/): Why t-SNE cluster sizes and distances must not be over-interpreted.
